In [6]:
import requests
from bs4 import BeautifulSoup
import gradio as gr
import ollama
import json
import re
from datetime import datetime


g:\python projects\research engine\Ai-Research-Bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
OLLAMA_MODEL = "qwen3:8b"

def ask_ollama(prompt: str, system: str = None) -> str:
    """Small wrapper around Ollama's chat API so every tool calls it the same way."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(model=OLLAMA_MODEL, messages=messages)
    return response["message"]["content"]

# quick check
print(ask_ollama("Reply with just the word 'ready' if you can read this."))

ready


In [8]:
def generate_research_questions(topic):
    prompt = f"""
Generate 5 specific and useful research questions or search queries about the following topic:

{topic}

Requirements:
- Each question/query must be on a separate line.
- Do not number them.
- Do not use bullet points.
- Do not add any introduction or explanation.
- Make the questions specific enough to be useful for web research.
"""
    
    response = ask_ollama(prompt)
    
    questions = []
    for line in response.splitlines():
        line = re.sub(r"^\s*[-*•\d.)]+\s*", "", line).strip()
        if line:
            questions.append(line)
    
    return questions

In [9]:
generate_research_questions("Solar Energy")

['How do perovskite solar cells compare to traditional silicon-based cells in terms of efficiency, durability, and manufacturing costs?',
 'What are the key environmental and economic trade-offs associated with large-scale solar panel production and disposal?',
 'How do government subsidies and tax incentives influence the adoption rate of residential solar energy systems in different countries?',
 'What are the current technological challenges in improving the energy storage capacity of solar-powered batteries for grid-scale applications?',
 'How does the efficiency of solar energy systems vary in regions with different levels of solar irradiance and temperature fluctuations?']

In [10]:
def summarize_source(content):
    prompt = f"""
Summarize the following source in 3-5 clear sentences.

Requirements:
- Use only information explicitly stated in the source.
- Do not add outside information or make unsupported claims.
- Focus on the main ideas, important facts, and findings.
- Keep the summary concise and factual.

Source:
{content}
"""
    
    return ask_ollama(prompt).strip()

In [11]:
test_content = """
Solar energy is a renewable source of energy that comes from sunlight.
Solar panels use photovoltaic cells to convert sunlight into electricity.
Solar power can reduce dependence on fossil fuels and lower greenhouse gas emissions.
However, solar energy production depends on sunlight and can require significant initial investment.
"""

In [12]:
summarize_source(test_content)

'Solar energy is a renewable energy source derived from sunlight. Solar panels utilize photovoltaic cells to convert sunlight into electricity. It can decrease reliance on fossil fuels and reduce greenhouse gas emissions. However, its production depends on sunlight availability and requires substantial initial investment.'

In [13]:
def compare_sources(source1, source2):
    prompt = f"""
Compare the following two sources.

Your response must contain exactly two sections:

Agreements:
- List the main points that both sources agree on.

Differences:
- List the main points where the sources disagree, differ in emphasis, or provide different information.

Do not add information that is not present in either source.

Source 1:
{source1}

Source 2:
{source2}
"""
    
    return ask_ollama(prompt).strip()

In [14]:
source1 = """
Solar energy is a renewable source of energy.
Solar panels can reduce dependence on fossil fuels.
The main challenge is that solar power depends on sunlight.
"""

source2 = """
Solar power is a clean and renewable energy source.
Using solar panels can reduce the use of fossil fuels.
Solar energy production can be affected by weather and the availability of sunlight.
"""

In [15]:
compare_sources(source1, source2)

'**Agreements:**  \n- Both sources state that solar energy is a renewable source of energy.  \n- Both mention that solar panels can reduce reliance on fossil fuels.  \n\n**Differences:**  \n- Source 1 refers to the main challenge as dependence on sunlight, while Source 2 emphasizes that solar energy production is affected by weather and sunlight availability.  \n- Source 2 explicitly describes solar power as "clean," a term not used in Source 1.'

In [16]:
def generate_report(sources, topic):
    sources_text = "\n\n".join(
        f"Source {i + 1}:\n{source}" 
        for i, source in enumerate(sources)
    )

    prompt = f"""
Write a structured research report about the following topic:

{topic}

Use only the information provided in the source summaries below.
Do not invent facts or add information from outside the sources.

The report must contain exactly these sections:

1. Introduction
Briefly introduce the research topic and its context.

2. Key Findings
Present the most important findings from the sources in a clear and organized way.

3. Conclusion
Summarize the main conclusions based on the provided sources.

4. Sources
List the provided sources as Source 1, Source 2, etc.

Source summaries:
{sources_text}
"""
    
    return ask_ollama(prompt).strip()

In [17]:
test_sources = [
    """
    Solar energy is a renewable source of energy.
    Solar panels convert sunlight into electricity.
    Solar power can reduce dependence on fossil fuels.
    """,

    """
    Solar energy is clean and renewable.
    Advances in photovoltaic technology have improved solar panel efficiency.
    Solar energy production depends on sunlight and weather conditions.
    """,

    """
    Solar power has environmental benefits because it can reduce greenhouse gas emissions.
    However, solar installations can require significant initial investment.
    """
]

In [18]:
generate_report(test_sources, "Solar Energy")

'1. Introduction  \nSolar energy is a renewable energy source that has gained significant attention as a sustainable alternative to traditional fossil fuels. It leverages sunlight to generate electricity and heat, offering potential solutions to reduce environmental impact and energy dependency. This report summarizes key findings from provided sources to explore the characteristics, benefits, and challenges of solar energy.  \n\n2. Key Findings  \n- **Renewable and Clean Energy**: Solar energy is a renewable and clean energy source, producing electricity without emitting greenhouse gases (Source 1, Source 2).  \n- **Technological Advancements**: Advances in photovoltaic technology have improved the efficiency of solar panels, making solar power more viable and cost-effective (Source 2).  \n- **Environmental Benefits**: Solar power reduces greenhouse gas emissions, contributing to climate change mitigation (Source 3).  \n- **Dependence on Weather**: Solar energy production relies on su